Este notebook toma los EPT existentes para cada vendor-dow-block y los ajusta según las instrucciones vigentes de `to_adjust_now`.

La hoja de input se mantiene como una tabla de instrucciones. El detalle estandarizado por vendor se escribe por separado en `results_export`.


INPUT:
* `to_adjust_now`: `vendor_code` + `ept_new`; opcionales `franchise_id`, `wave` y `flag`.
* También acepta los aliases `Vendor code` y `EPT min` cuando no existen los nombres estándar.
* `EPT actuales`: resultados de la query por vendor-dow-block.

OUTPUT:
* CSV TES en Drive.
* `results_export`: una fila por vendor efectivamente exportado, con origen, alcance y detalle del ajuste.
* `last_executed_at` se actualiza en `to_adjust_now` solo al finalizar correctamente.


In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Data Load

### instrucciones: `to_adjust_now`


In [2]:
reductions_only = False
required_diff_mins = 3

#### waves calculados

In [3]:
# # waves calculator
# file_name = 'new_preps_mod.csv'

# from google.colab import drive
# drive.mount('/content/drive')
# file_path_main = '/content/drive/MyDrive/lower ept y awt/ept reductions input data/'
# file_path = file_path_main + file_name
# new_preps = pd.read_csv(file_path)
# new_preps.head(3)

#### input unificado: waves y ajustes particulares


In [4]:
import re
import numpy as np
import pandas as pd
import gspread
from google.colab import auth
import google.auth

auth.authenticate_user()

credentials, _ = google.auth.default(scopes=[
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
])

gc = gspread.authorize(credentials)

sheet_id = "1WXZXGzf-nfwJgdt7k2WqOnh7ys5W_JqXqhuPi2VJ-IE"
INPUT_SHEET_NAME = "to_adjust_now"
RESULTS_SHEET_NAME = "results_export"

spreadsheet = gc.open_by_key(sheet_id)
input_worksheet = spreadsheet.worksheet(INPUT_SHEET_NAME)
input_values = input_worksheet.get_all_values()

if not input_values or not input_values[0]:
    raise ValueError(f"La hoja '{INPUT_SHEET_NAME}' está vacía.")

input_headers_original = [str(column).strip() for column in input_values[0]]

if len(input_headers_original) != len(set(input_headers_original)):
    duplicates = pd.Series(input_headers_original).value_counts()
    duplicates = duplicates[duplicates > 1].index.tolist()
    raise ValueError(
        "Hay encabezados repetidos en to_adjust_now: "
        + ", ".join(map(str, duplicates))
    )

input_table = pd.DataFrame(
    input_values[1:],
    columns=input_headers_original
)
input_table["_input_row_number"] = range(2, len(input_table) + 2)

# Ignorar filas completamente vacías, sin considerar timestamps anteriores.
content_columns = [
    column for column in input_headers_original
    if column != "last_executed_at"
]

if content_columns:
    has_content = input_table[content_columns].apply(
        lambda row: row.astype("string").str.strip().ne("").any(),
        axis=1
    )
    input_table = input_table.loc[has_content].copy()

if input_table.empty:
    raise ValueError(f"La hoja '{INPUT_SHEET_NAME}' no tiene instrucciones.")


def normalized_header(value):
    return re.sub(r"[\s_-]+", " ", str(value).strip()).casefold()


def use_column_alias(df, target, aliases):
    """Usa un alias solo si falta la columna estándar; si ambas existen, completa vacíos."""
    lookup = {normalized_header(column): column for column in df.columns}
    alias_column = next(
        (
            lookup[normalized_header(alias)]
            for alias in aliases
            if normalized_header(alias) in lookup
            and lookup[normalized_header(alias)] != target
        ),
        None
    )

    if target not in df.columns:
        if alias_column is None:
            return df
        return df.rename(columns={alias_column: target})

    if alias_column is not None:
        target_blank = (
            df[target].astype("string").str.strip().isin(["", "nan", "<NA>"])
            | df[target].isna()
        )
        df.loc[target_blank, target] = df.loc[target_blank, alias_column]

    return df


column_aliases = {
    "vendor_code": ["Vendor code", "Vendor Code", "vendor code"],
    "ept_new": ["EPT min", "EPT Min", "ept min"],
    "franchise_id": ["Franchise ID", "Franchise id", "franchise id"],
    "wave": ["Wave"],
    "flag": ["Flag"],
}

for standard_name, aliases in column_aliases.items():
    input_table = use_column_alias(input_table, standard_name, aliases)

if "ept_new" not in input_table.columns:
    raise ValueError(
        "Falta la columna ept_new. También se acepta el alias 'EPT min'."
    )

# vendor_code puede faltar si todas las instrucciones vienen por franchise_id.
for optional_column in [
    "vendor_code", "franchise_id", "wave", "flag", "last_executed_at"
]:
    if optional_column not in input_table.columns:
        input_table[optional_column] = pd.NA

new_preps = input_table.copy()
input_rows_to_mark = input_table["_input_row_number"].astype(int).tolist()

execution_timestamp = pd.Timestamp.now(tz="America/Santiago").floor("s")
executed_at = execution_timestamp.strftime("%Y-%m-%d %H:%M:%S")
execution_id = execution_timestamp.strftime("%Y%m%d_%H%M%S")

print(
    f"Instrucciones leídas: {len(new_preps):,} | "
    f"Ejecución: {executed_at} America/Santiago"
)

new_preps.head()


Instrucciones leídas: 9 | Ejecución: 2026-09-04 17:27:09 America/Santiago


,wave,vendor_code,flag,ept_new,last_executed_at,_input_row_number,franchise_id
0,,616172,Manual modification,25,,2,<NA>
1,,573294,Manual modification,25,,3,<NA>
2,,542387,Manual modification,30,,4,<NA>
3,,100004,Manual modification,15,,5,<NA>
4,,597098,Manual modification,23,,6,<NA>


In [5]:
def normalize_id(series):
    normalized = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    return normalized.mask(
        normalized.isna()
        | normalized.str.lower().isin(["", "nan", "none", "<na>"])
    )


new_preps["vendor_code"] = normalize_id(new_preps["vendor_code"])
new_preps["franchise_id"] = normalize_id(new_preps["franchise_id"])
new_preps["ept_new"] = pd.to_numeric(new_preps["ept_new"], errors="coerce")

for column in ["wave", "flag"]:
    new_preps[column] = (
        new_preps[column]
        .astype("string")
        .str.strip()
        .mask(lambda values: values.isna() | values.eq(""))
    )

rows_without_target = new_preps[
    new_preps["vendor_code"].isna()
    & new_preps["franchise_id"].isna()
]

if not rows_without_target.empty:
    raise ValueError(
        "Hay filas en to_adjust_now sin vendor_code ni franchise_id: "
        f"{rows_without_target['_input_row_number'].astype(int).tolist()}"
    )

rows_without_ept = new_preps[new_preps["ept_new"].isna()]

if not rows_without_ept.empty:
    raise ValueError(
        "Hay filas en to_adjust_now con ept_new vacío o no numérico: "
        f"{rows_without_ept['_input_row_number'].astype(int).tolist()}"
    )

# Si existe vendor_code, la instrucción es individual aunque también venga
# franchise_id. Una fila sin vendor_code se interpreta como franquicia completa.
new_preps["adjustment_scope"] = np.where(
    new_preps["vendor_code"].notna(),
    "vendor",
    "franchise"
)
new_preps["instruction_key"] = np.where(
    new_preps["adjustment_scope"].eq("vendor"),
    "vendor:" + new_preps["vendor_code"].astype("string"),
    "franchise:" + new_preps["franchise_id"].astype("string")
)

# No ocultar conflictos eligiendo silenciosamente el menor EPT.
conflicts = (
    new_preps.groupby("instruction_key")["ept_new"]
    .nunique(dropna=False)
)
conflicts = conflicts[conflicts > 1]

if not conflicts.empty:
    raise ValueError(
        "Hay instrucciones repetidas con distintos ept_new: "
        + ", ".join(conflicts.index.tolist())
    )

# Para duplicados idénticos se conserva la última fila de la hoja, incluyendo
# sus metadatos wave y flag. Las franquicias ya no colapsan por vendor_code vacío.
new_preps = (
    new_preps
    .sort_values("_input_row_number")
    .drop_duplicates("instruction_key", keep="last")
    .reset_index(drop=True)
)

print(
    f"Reglas válidas: {len(new_preps):,} | "
    f"vendors: {(new_preps['adjustment_scope'] == 'vendor').sum():,} | "
    f"franquicias: {(new_preps['adjustment_scope'] == 'franchise').sum():,}"
)

new_preps.head()


Reglas válidas: 9 | vendors: 9 | franquicias: 0


,wave,vendor_code,flag,ept_new,last_executed_at,_input_row_number,franchise_id,adjustment_scope,instruction_key
0,<NA>,616172,Manual modification,25,,2,<NA>,vendor,vendor:616172
1,<NA>,573294,Manual modification,25,,3,<NA>,vendor,vendor:573294
2,<NA>,542387,Manual modification,30,,4,<NA>,vendor,vendor:542387
3,<NA>,100004,Manual modification,15,,5,<NA>,vendor,vendor:100004
4,<NA>,597098,Manual modification,23,,6,<NA>,vendor,vendor:597098


#### generate query

In [6]:
from datetime import timedelta
import pandas as pd

# ============================================================
# 1. IDs únicos para limitar la query
# ============================================================

vendor_codes = (
    new_preps.loc[
        new_preps["adjustment_scope"].eq("vendor"),
        "vendor_code"
    ]
    .dropna()
    .drop_duplicates()
    .tolist()
)

franchise_ids = (
    new_preps.loc[
        new_preps["adjustment_scope"].eq("franchise"),
        "franchise_id"
    ]
    .dropna()
    .drop_duplicates()
    .tolist()
)


def sql_string_list(values):
    return ",\n      ".join(
        f"'{value.replace(chr(39), chr(39) * 2)}'"
        for value in values
    )


target_conditions = []

if vendor_codes:
    target_conditions.append(
        "CAST(p.partner_id AS STRING) IN (\n"
        f"      {sql_string_list(vendor_codes)}\n"
        "    )"
    )

if franchise_ids:
    target_conditions.append(
        "CAST(p.franchise.franchise_id AS STRING) IN (\n"
        f"      {sql_string_list(franchise_ids)}\n"
        "    )"
    )

if not target_conditions:
    raise ValueError(
        "to_adjust_now no contiene vendor_code ni franchise_id válidos."
    )

target_filter_sql = "\n    OR ".join(target_conditions)

# ============================================================
# 2. Últimos 7 días completos terminando ayer, hora de Chile
# ============================================================

today = pd.Timestamp.now(tz="America/Santiago").date()
start_date = today - timedelta(days=7)
end_date = today - timedelta(days=1)


# ============================================================
# 3. Query
# ============================================================

query = f"""
WITH hdm AS (
  SELECT
    SAFE_CAST(order_code AS INT64) AS order_code,

    high_demand_mode.is_hd_order AS hd_order,
    SAFE_CAST(
      high_demand_mode.minutes_added AS FLOAT64
    ) AS minutes_added,
    SAFE_CAST(
      high_demand_mode.duration AS FLOAT64
    ) AS duration,

    high_demand_mode.enabled_at,
    high_demand_mode.disabled_at

  FROM `fulfillment-dwh-production.curated_data_shared_vendor.growth_vendor_orders`

  WHERE created_date BETWEEN DATE '{start_date}'
                         AND DATE '{end_date}'
    AND country_code = 'cl'
    AND high_demand_mode.is_hd_order IS TRUE
    AND requested_pickup_at IS NULL
),

target_stores AS (
  SELECT DISTINCT
    CAST(p.partner_id AS STRING) AS vendor_code,
    p.partner_name AS store_name,

    CAST(
      p.franchise.franchise_id AS STRING
    ) AS franchise_id,

    COALESCE(
      CAST(p.franchise.franchise_id AS STRING),
      CONCAT(
        'VENDOR_',
        CAST(p.partner_id AS STRING)
      )
    ) AS franchise_group_id,

    p.franchise.franchise_name AS franchise_name

  FROM `peya-bi-tools-pro.il_core.dim_partner` p

  WHERE p.is_online = TRUE
    AND (
      {target_filter_sql}
    )
),

base AS (
  SELECT
    l.peya_order_id,
    l.vendor.vendor_code AS vendor_code,

    COALESCE(
      p.store_name,
      l.vendor.name
    ) AS store_name,

    l.city.city_name AS city_name,

    p.franchise_id,
    p.franchise_group_id,
    p.franchise_name,

    l.food_is_ready_at,
    l.vendor.vertical_type,

    CASE
      WHEN l.food_is_ready_at IS NOT NULL
      THEN TIMESTAMP_DIFF(
        l.food_is_ready_at,
        TIMESTAMP(l.created_at_local),
        SECOND
      )
    END AS fir_seconds,

    DATE(l.created_date_local) AS order_date,

    EXTRACT(
      DAYOFWEEK
      FROM DATE(l.created_date_local)
    ) AS day_of_week_num,

    FORMAT_DATE(
      '%A',
      DATE(l.created_date_local)
    ) AS day_of_week_name,

    CASE
      WHEN EXTRACT(HOUR FROM l.created_at_local)
        BETWEEN 12 AND 14
        THEN 'lunch'

      WHEN EXTRACT(HOUR FROM l.created_at_local)
        BETWEEN 19 AND 21
        THEN 'dinner'

      ELSE 'valle'
    END AS time_block,

    SAFE_DIVIDE(
      l.estimated_prep_time,
      60
    ) AS ept_min,

    SAFE_DIVIDE(
      l.timings.avoidable_wait_time,
      60
    ) AS awt_min,

    TIMESTAMP_DIFF(
      d.rider_picked_up_at_local,
      l.created_at_local,
      MINUTE
    ) AS created_to_pickup,

    o.is_slow_order AS slow,
    o.non_seamless_order AS non_seamless,

    h.order_code AS hd_order_code,
    h.minutes_added AS hd_minutes_added,
    h.duration AS hd_duration,
    h.enabled_at AS hd_enabled_at,
    h.disabled_at AS hd_disabled_at

  FROM `peya-bi-tools-pro.il_logistics.fact_logistic_orders` l

  LEFT JOIN UNNEST(l.deliveries) d

  INNER JOIN target_stores p
    ON l.vendor.vendor_code = p.vendor_code

  LEFT JOIN `peya-datamarts-pro.dm_fulfillment.non_seamless_delivery_order_level` o
    ON o.platform_order_code = l.peya_order_id

  LEFT JOIN hdm h
    ON h.order_code = SAFE_CAST(
      l.peya_order_id AS INT64
    )

  WHERE l.country_code = 'cl'
    -- AND l.vendor.vertical_type = 'restaurants'
    AND l.is_preorder = FALSE

    AND l.created_date_local
      BETWEEN DATE '{start_date}'
          AND DATE '{end_date}'
),

zero_food_ready_franchises AS (
  SELECT
    franchise_group_id,
    franchise_id,
    franchise_name,

    COUNT(
      DISTINCT peya_order_id
    ) AS franchise_orders,

    COUNT(
      DISTINCT vendor_code
    ) AS franchise_stores,

    COUNT(
      DISTINCT IF(
        food_is_ready_at IS NOT NULL,
        peya_order_id,
        NULL
      )
    ) AS franchise_orders_with_fir,

    SAFE_DIVIDE(
      COUNT(
        DISTINCT IF(
          food_is_ready_at IS NOT NULL,
          peya_order_id,
          NULL
        )
      ),
      COUNT(DISTINCT peya_order_id)
    ) AS franchise_fir

  FROM base

  GROUP BY
    franchise_group_id,
    franchise_id,
    franchise_name
),

valid_stores AS (
  SELECT
    b.franchise_group_id,
    b.franchise_id,
    b.franchise_name,
    b.city_name,
    b.vertical_type,

    z.franchise_orders,
    z.franchise_stores,
    z.franchise_orders_with_fir,
    z.franchise_fir,

    b.vendor_code,
    b.store_name,

    COUNT(
      DISTINCT b.peya_order_id
    ) AS store_orders_total,

    COUNT(
      DISTINCT IF(
        b.food_is_ready_at IS NOT NULL,
        b.peya_order_id,
        NULL
      )
    ) AS store_orders_with_fir,

    SAFE_DIVIDE(
      COUNT(
        DISTINCT IF(
          b.food_is_ready_at IS NOT NULL,
          b.peya_order_id,
          NULL
        )
      ),
      COUNT(DISTINCT b.peya_order_id)
    ) AS store_fir

  FROM base b

  INNER JOIN zero_food_ready_franchises z
    ON b.franchise_group_id = z.franchise_group_id

  GROUP BY
    b.franchise_group_id,
    b.franchise_id,
    b.franchise_name,
    b.city_name,
    b.vertical_type,

    z.franchise_orders,
    z.franchise_stores,
    z.franchise_orders_with_fir,
    z.franchise_fir,

    b.vendor_code,
    b.store_name
),

store_day_block_metrics AS (
  SELECT
    b.franchise_group_id,
    b.franchise_id,
    b.vendor_code,
    b.city_name,
    b.vertical_type,

    b.day_of_week_num,
    b.day_of_week_name,
    b.time_block,

    SUM(
      b.fir_seconds
    ) AS sum_fir_seconds,

    ROUND(
      SAFE_DIVIDE(
        SUM(b.fir_seconds),
        60
      ),
      2
    ) AS sum_fir_min,

    ROUND(
      SAFE_DIVIDE(
        AVG(b.fir_seconds),
        60
      ),
      2
    ) AS avg_fir_min,

    COUNT(
      DISTINCT b.peya_order_id
    ) AS total_orders,

    COUNT(
      DISTINCT IF(
        b.slow = 1,
        b.peya_order_id,
        NULL
      )
    ) AS slow_orders,

    COUNT(
      DISTINCT IF(
        b.non_seamless = TRUE,
        b.peya_order_id,
        NULL
      )
    ) AS non_seamless_orders,

    COUNT(
      DISTINCT IF(
        b.food_is_ready_at IS NOT NULL,
        b.peya_order_id,
        NULL
      )
    ) AS orders_with_fir,

    SAFE_DIVIDE(
      COUNT(
        DISTINCT IF(
          b.food_is_ready_at IS NOT NULL,
          b.peya_order_id,
          NULL
        )
      ),
      COUNT(DISTINCT b.peya_order_id)
    ) AS fir,

    ROUND(
      AVG(b.ept_min),
      2
    ) AS ept,

    ROUND(
      AVG(b.awt_min),
      2
    ) AS awt,

    ROUND(
      AVG(b.ept_min) + AVG(b.awt_min),
      2
    ) AS tt,

    ROUND(
      AVG(b.created_to_pickup),
      2
    ) AS ctp

  FROM base b

  INNER JOIN valid_stores s
    ON b.franchise_group_id = s.franchise_group_id
   AND b.vendor_code = s.vendor_code
   AND b.city_name = s.city_name
   AND b.vertical_type = s.vertical_type

  GROUP BY
    b.franchise_group_id,
    b.franchise_id,
    b.vendor_code,
    b.city_name,
    b.vertical_type,
    b.day_of_week_num,
    b.day_of_week_name,
    b.time_block
),

hdm_order_metrics AS (
  SELECT
    franchise_group_id,
    franchise_id,
    vendor_code,
    city_name,
    vertical_type,

    day_of_week_num,
    day_of_week_name,
    time_block,

    COUNT(
      DISTINCT peya_order_id
    ) AS hd_total_orders,

    ROUND(
      SUM(hd_minutes_added),
      2
    ) AS hd_total_minutes_added

  FROM (
    SELECT
      franchise_group_id,
      franchise_id,
      vendor_code,
      city_name,
      vertical_type,

      day_of_week_num,
      day_of_week_name,
      time_block,

      peya_order_id,

      MAX(
        COALESCE(
          hd_minutes_added,
          0
        )
      ) AS hd_minutes_added

    FROM base

    WHERE hd_order_code IS NOT NULL

    GROUP BY
      franchise_group_id,
      franchise_id,
      vendor_code,
      city_name,
      vertical_type,
      day_of_week_num,
      day_of_week_name,
      time_block,
      peya_order_id
  )

  GROUP BY
    franchise_group_id,
    franchise_id,
    vendor_code,
    city_name,
    vertical_type,
    day_of_week_num,
    day_of_week_name,
    time_block
),

hdm_trigger_metrics AS (
  SELECT
    franchise_group_id,
    franchise_id,
    vendor_code,
    city_name,
    vertical_type,

    day_of_week_num,
    day_of_week_name,
    time_block,

    COUNT(*) AS started_triggers,

    ROUND(
      AVG(duration),
      2
    ) AS avg_trigger_duration_min

  FROM (
    SELECT
      franchise_group_id,
      franchise_id,
      vendor_code,
      city_name,
      vertical_type,

      EXTRACT(
        DAYOFWEEK
        FROM DATE(hd_enabled_at)
      ) AS day_of_week_num,

      FORMAT_DATE(
        '%A',
        DATE(hd_enabled_at)
      ) AS day_of_week_name,

      CASE
        WHEN EXTRACT(HOUR FROM hd_enabled_at)
          BETWEEN 12 AND 14
          THEN 'lunch'

        WHEN EXTRACT(HOUR FROM hd_enabled_at)
          BETWEEN 19 AND 21
          THEN 'dinner'

        ELSE 'valle'
      END AS time_block,

      hd_enabled_at,

      MAX(
        hd_duration
      ) AS duration

    FROM base

    WHERE hd_enabled_at IS NOT NULL

    GROUP BY
      franchise_group_id,
      franchise_id,
      vendor_code,
      city_name,
      vertical_type,
      day_of_week_num,
      day_of_week_name,
      time_block,
      hd_enabled_at
  )

  GROUP BY
    franchise_group_id,
    franchise_id,
    vendor_code,
    city_name,
    vertical_type,
    day_of_week_num,
    day_of_week_name,
    time_block
)

SELECT
  t.franchise_id,
  t.franchise_name,
  s.city_name,
  s.vertical_type,

  s.franchise_orders,
  s.franchise_stores,
  s.franchise_orders_with_fir,

  ROUND(
    s.franchise_fir,
    4
  ) AS franchise_fir,

  t.vendor_code,

  COALESCE(
    s.store_name,
    t.store_name
  ) AS store_name,

  s.store_orders_total,
  s.store_orders_with_fir,

  ROUND(
    s.store_fir,
    4
  ) AS store_fir,

  m.day_of_week_num,
  m.day_of_week_name,
  m.time_block,

  m.total_orders,
  m.slow_orders,
  m.non_seamless_orders,

  COALESCE(
    hdo.hd_total_orders,
    0
  ) AS hd_total_orders,

  COALESCE(
    hdo.hd_total_minutes_added,
    0
  ) AS hd_total_minutes_added,

  COALESCE(
    hdt.started_triggers,
    0
  ) AS started_triggers,

  hdt.avg_trigger_duration_min,

  m.orders_with_fir,

  ROUND(
    m.fir,
    4
  ) AS fir_ratio,

  m.ept,
  m.awt,
  m.tt,
  m.ctp,
  m.avg_fir_min AS fir

FROM target_stores t

LEFT JOIN valid_stores s
  ON t.franchise_group_id = s.franchise_group_id
 AND t.vendor_code = s.vendor_code

LEFT JOIN store_day_block_metrics m
  ON t.franchise_group_id = m.franchise_group_id
 AND s.vendor_code = m.vendor_code
 AND s.city_name = m.city_name
 AND s.vertical_type = m.vertical_type

LEFT JOIN hdm_order_metrics hdo
  ON s.franchise_group_id = hdo.franchise_group_id
 AND s.vendor_code = hdo.vendor_code
 AND s.city_name = hdo.city_name
 AND s.vertical_type = hdo.vertical_type
 AND m.day_of_week_num = hdo.day_of_week_num
 AND m.time_block = hdo.time_block

LEFT JOIN hdm_trigger_metrics hdt
  ON s.franchise_group_id = hdt.franchise_group_id
 AND s.vendor_code = hdt.vendor_code
 AND s.city_name = hdt.city_name
 AND s.vertical_type = hdt.vertical_type
 AND m.day_of_week_num = hdt.day_of_week_num
 AND m.time_block = hdt.time_block

ORDER BY
  t.franchise_name,
  COALESCE(
    s.store_name,
    t.store_name
  ),
  m.day_of_week_num,

  CASE m.time_block
    WHEN 'lunch' THEN 1
    WHEN 'dinner' THEN 2
    WHEN 'valle' THEN 3
  END
"""


# print(
#     f"Reglas individuales: {len(vendor_codes)} | "
#     f"Reglas por franquicia: {len(franchise_ids)} | "
#     f"Fechas: {start_date} a {end_date}"
# )

print(query)


WITH hdm AS (
  SELECT
    SAFE_CAST(order_code AS INT64) AS order_code,

    high_demand_mode.is_hd_order AS hd_order,
    SAFE_CAST(
      high_demand_mode.minutes_added AS FLOAT64
    ) AS minutes_added,
    SAFE_CAST(
      high_demand_mode.duration AS FLOAT64
    ) AS duration,

    high_demand_mode.enabled_at,
    high_demand_mode.disabled_at

  FROM `fulfillment-dwh-production.curated_data_shared_vendor.growth_vendor_orders`

  WHERE created_date BETWEEN DATE '2026-08-28'
                         AND DATE '2026-09-03'
    AND country_code = 'cl'
    AND high_demand_mode.is_hd_order IS TRUE
    AND requested_pickup_at IS NULL
),

target_stores AS (
  SELECT DISTINCT
    CAST(p.partner_id AS STRING) AS vendor_code,
    p.partner_name AS store_name,

    CAST(
      p.franchise.franchise_id AS STRING
    ) AS franchise_id,

    COALESCE(
      CAST(p.franchise.franchise_id AS STRING),
      CONCAT(
        'VENDOR_',
        CAST(p.partner_id AS STRING)
      )
    ) AS franchise_

In [7]:
# input_confirmation = input("¿Ya tomaste los resultados de la query y actualizaste los EPT actuales en la hoja de cálculo?  \n https://docs.google.com/spreadsheets/d/1WXZXGzf-nfwJgdt7k2WqOnh7ys5W_JqXqhuPi2VJ-IE \n Escribe 'sí' para continuar: ")

# if input_confirmation.lower() != 'si':
#     raise ValueError("Por favor, actualiza los EPTs en la hoja de cálculo y vuelve a ejecutar esta celda.")

# print("Confirmación recibida. Continuando con el procesamiento.")

### ept_actuales

In [8]:
# # EPT actuales
# file_name = 'Wave week36.csv'

# from google.colab import drive
# drive.mount('/content/drive')
# file_path_main = '/content/drive/MyDrive/lower ept y awt/ept reductions input data/'
# file_path = file_path_main + file_name
# ept_actuales_raw = pd.read_csv(file_path)
# ept_actuales_raw.head(3)

In [9]:
import pandas as pd
project_id = 'peya-chile'

# Ejecuta la consulta en BigQuery y descarga los datos a Colab
ept_actuales_raw = pd.read_gbq(query, project_id=project_id)

# Muestra las primeras 5 filas del resultado
ept_actuales_raw.head()

/tmp/ipykernel_4417/1760443758.py:5: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  ept_actuales_raw = pd.read_gbq(query, project_id=project_id)


,franchise_id,franchise_name,city_name,vertical_type,franchise_orders,franchise_stores,franchise_orders_with_fir,franchise_fir,vendor_code,store_name,store_orders_total,store_orders_with_fir,store_fir,day_of_week_num,day_of_week_name,time_block,total_orders,slow_orders,non_seamless_orders,hd_total_orders,hd_total_minutes_added,started_triggers,avg_trigger_duration_min,orders_with_fir,fir_ratio,ept,awt,tt,ctp,fir
0,None,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,2,Monday,dinner,1,0,0,0,0.0,0,NaN,1,1.0,13.0,5.10,18.10,19.00,256.05
1,None,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,2,Monday,valle,2,0,0,0,0.0,0,NaN,1,0.5,13.0,8.23,21.23,24.00,258.25
2,None,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,3,Tuesday,lunch,1,0,0,0,0.0,0,NaN,1,1.0,13.0,NaN,NaN,24.00,263.47
3,None,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,3,Tuesday,dinner,3,1,1,0,0.0,0,NaN,3,1.0,13.0,16.72,29.72,37.67,276.97
4,None,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,3,Tuesday,valle,1,0,0,1,8.0,0,NaN,1,1.0,13.0,15.08,28.08,29.00,267.12


In [11]:
# # 1. Prepara el DataFrame: gspread no acepta valores NaN ni formatos complejos de fecha
# df_to_export = ept_actuales_raw.fillna("")
# df_to_export = df_to_export.astype(str) # Convertimos todo a texto para evitar errores de formato

# # 2. Abre el documento de Google Sheets (usa el mismo sheet_id que ya tienes u otro)
# # gc ya está autorizado en las celdas anteriores
# sheet_id = "TU_ID_DEL_DOCUMENTO_AQUI"
# spreadsheet = gc.open_by_key(sheet_id)

# # 3. Crea una nueva pestaña para los resultados de la query
# nombre_pestaña = "Resultados Query"
# try:
#     worksheet = spreadsheet.worksheet(nombre_pestaña)
#     worksheet.clear() # Limpia la pestaña si ya existe
# except gspread.WorksheetNotFound:
#     worksheet = spreadsheet.add_worksheet(title=nombre_pestaña, rows=100, cols=20)

# # 4. Convierte el DataFrame a una lista de listas (encabezados + valores)
# data = [df_to_export.columns.values.tolist()] + df_to_export.values.tolist()

# # 5. Escribe los datos en la hoja
# worksheet.update(range_name="A1", values=data, value_input_option="RAW")
# print(f"¡Exportación exitosa a la pestaña '{nombre_pestaña}'!")

In [12]:
# import gspread
# from google.colab import auth
# import google.auth

# auth.authenticate_user()

# credentials, _ = google.auth.default(scopes=[
#     "https://www.googleapis.com/auth/spreadsheets",
#     "https://www.googleapis.com/auth/drive"
# ])

# gc = gspread.authorize(credentials)
# sheet_id = "1WXZXGzf-nfwJgdt7k2WqOnh7ys5W_JqXqhuPi2VJ-IE"
# # manual ept modifications
# # https://docs.google.com/spreadsheets/d/1WXZXGzf-nfwJgdt7k2WqOnh7ys5W_JqXqhuPi2VJ-IE/edit?gid=0#gid=0
# sheet_name = "EPT actuales"

# worksheet = gc.open_by_key(sheet_id).worksheet(sheet_name)
# data = worksheet.get_all_values()
# ept_actuales_raw = pd.DataFrame(data[1:], columns=data[0])

# ept_actuales_raw.head()

In [13]:
ept_actuales = ept_actuales_raw.copy()
ept_actuales.head(3) # agregar a query los filtros de la query anterior, incluir city_name, slow_orders, non_seamless_orders, ctp

,franchise_id,franchise_name,city_name,vertical_type,franchise_orders,franchise_stores,franchise_orders_with_fir,franchise_fir,vendor_code,store_name,store_orders_total,store_orders_with_fir,store_fir,day_of_week_num,day_of_week_name,time_block,total_orders,slow_orders,non_seamless_orders,hd_total_orders,hd_total_minutes_added,started_triggers,avg_trigger_duration_min,orders_with_fir,fir_ratio,ept,awt,tt,ctp,fir
0,None,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,2,Monday,dinner,1,0,0,0,0.0,0,NaN,1,1.0,13.0,5.10,18.10,19.0,256.05
1,None,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,2,Monday,valle,2,0,0,0,0.0,0,NaN,1,0.5,13.0,8.23,21.23,24.0,258.25
2,None,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,3,Tuesday,lunch,1,0,0,0,0.0,0,NaN,1,1.0,13.0,NaN,NaN,24.0,263.47


#### vendors validation

In [14]:
# Validar que las reglas por franquicia sí se hayan podido expandir.
# Los vendor_code individuales pueden no tener historia y se completan más adelante.
ept_actuales["vendor_code"] = normalize_id(ept_actuales["vendor_code"])
ept_actuales["franchise_id"] = normalize_id(ept_actuales["franchise_id"])

franchises_expected = set(franchise_ids)
franchises_found = set(ept_actuales["franchise_id"].dropna())
missing_franchises = sorted(franchises_expected - franchises_found)

if missing_franchises:
    raise ValueError(
        "No se encontraron vendors online para estos franchise_id: "
        + ", ".join(missing_franchises)
    )

print(
    f"Franquicias expandidas: {len(franchises_expected)} | "
    f"Vendors obtenidos: {ept_actuales['vendor_code'].nunique()}"
)

Franquicias expandidas: 0 | Vendors obtenidos: 9


#### agrupado por vendor

In [15]:
import numpy as np

def wavg(x, value_col, weight_col='total_orders'):
    """
    Promedio ponderado usando únicamente filas con valor y peso válidos.
    """
    values = pd.to_numeric(x[value_col], errors='coerce')
    weights = pd.to_numeric(x[weight_col], errors='coerce')

    mask = values.notna() & weights.notna() & (weights > 0)

    if not mask.any() or weights.loc[mask].sum() == 0:
        return np.nan

    return np.average(
        values.loc[mask],
        weights=weights.loc[mask]
    )


def weighted_total(x, value_col, weight_col='total_orders'):
    """
    Convierte un promedio por fila en volumen total:
        sum(value * weight)
    """
    values = pd.to_numeric(x[value_col], errors='coerce')
    weights = pd.to_numeric(x[weight_col], errors='coerce')

    mask = values.notna() & weights.notna() & (weights > 0)

    if not mask.any():
        return 0.0

    return (values.loc[mask] * weights.loc[mask]).sum()

In [16]:
group_cols = [
    'vendor_code'
]

df_grouped = (
    ept_actuales
    .groupby(group_cols, as_index=False)
    .apply(lambda x: pd.Series({

        'franchise_id': x['franchise_id'].max(),
        'franchise_name': x['franchise_name'].max(),
        'city_name': x['city_name'].max(),
        'store_name': x['store_name'].max(),
        'franchise_orders': pd.to_numeric(x['franchise_orders'], errors='coerce').iloc[0],
        'franchise_stores': pd.to_numeric(x['franchise_stores'], errors='coerce').iloc[0],
        'store_orders_total': pd.to_numeric(x['store_orders_total'], errors='coerce').iloc[0],

        # Volúmenes - explicit conversion to numeric before summing
        'total_orders': pd.to_numeric(x['total_orders'], errors='coerce').sum(),
        'slow_orders': pd.to_numeric(x['slow_orders'], errors='coerce').sum(),
        'non_seamless_orders': pd.to_numeric(x['non_seamless_orders'], errors='coerce').sum(),
        'orders_with_fir': pd.to_numeric(x['orders_with_fir'], errors='coerce').sum(),

        # HDM - explicit conversion to numeric before summing
        # En la data raw la columna se llama hd_total_orders.
        'hd_orders': pd.to_numeric(x['hd_total_orders'], errors='coerce').sum(),
        'hd_total_minutes_added': pd.to_numeric(x['hd_total_minutes_added'], errors='coerce').sum(),

        # Total de minutos EPT antes de quitar HDM.
        'ept_total_minutes': weighted_total(
            x,
            value_col='ept',
            weight_col='total_orders'
        ),

        # Promedios
        'ept': wavg(x, 'ept', 'total_orders'),
        'awt': wavg(x, 'awt', 'total_orders'),
        'tt': wavg(x, 'tt', 'total_orders'),

        # FIR promedio entre las órdenes que efectivamente tienen FIR.
        # No se pondera por total_orders, sino por orders_with_fir.
        'fir': wavg(x, 'fir', 'orders_with_fir'),

        'ctp': wavg(x, 'ctp', 'total_orders')
    }), include_groups=False)
    .reset_index(drop=True)
)

df_grouped['slow_ratio'] = np.where(
    df_grouped['total_orders'] > 0,
    df_grouped['slow_orders'] / df_grouped['total_orders'],
    np.nan
)

df_grouped['non_seamless_ratio'] = np.where(
    df_grouped['total_orders'] > 0,
    df_grouped['non_seamless_orders'] / df_grouped['total_orders'],
    np.nan
)

# FIR ratio del vendor: share de órdenes que tienen FIR.
df_grouped['fir_ratio'] = np.where(
    df_grouped['total_orders'] > 0,
    df_grouped['orders_with_fir'] / df_grouped['total_orders'],
    np.nan
)

# Nombre explícito para el FIR promedio en minutos.
df_grouped['fir_avg'] = df_grouped['fir']

# EPT promedio quitando los minutos agregados por HDM:
# (sum(ept * total_orders) - sum(hd_total_minutes_added)) / sum(total_orders)
df_grouped['avg_ept_sin_hdm'] = np.where(
    df_grouped['total_orders'] > 0,
    (
        df_grouped['ept_total_minutes']
        - df_grouped['hd_total_minutes_added']
    ) / df_grouped['total_orders'],
    np.nan
)

# Control: diferencia promedio explicada por HDM.
df_grouped['avg_hdm_minutes_per_order'] = np.where(
    df_grouped['total_orders'] > 0,
    df_grouped['hd_total_minutes_added'] / df_grouped['total_orders'],
    np.nan
)

df_grouped = df_grouped.sort_values(
    ['franchise_name', 'store_name', 'city_name']
).reset_index(drop=True)


df_grouped['ept_raw'] = df_grouped['ept']
df_grouped['ept'] = df_grouped['avg_ept_sin_hdm']
df_grouped_raw = df_grouped.copy()

In [17]:
df_grouped_raw

,vendor_code,franchise_id,franchise_name,city_name,store_name,franchise_orders,franchise_stores,store_orders_total,total_orders,slow_orders,non_seamless_orders,orders_with_fir,hd_orders,hd_total_minutes_added,ept_total_minutes,ept,awt,tt,fir,ctp,slow_ratio,non_seamless_ratio,fir_ratio,fir_avg,avg_ept_sin_hdm,avg_hdm_minutes_per_order,ept_raw
0,100004,0016900002ZTfJUAA1,Montalbano,Calama,Schopdog Calama,75,1,75,75,8,16,72,14,112.0,1021.02,12.120267,5.188194,18.840972,251.960278,24.506667,0.106667,0.213333,0.960000,251.960278,12.120267,1.493333,13.613600
1,168677,<NA>,NaN,Valdivia,Cocavi Delivery,25,1,25,25,4,7,14,2,14.0,331.00,12.680000,13.989565,27.250435,270.977857,33.750000,0.160000,0.280000,0.560000,270.977857,12.680000,0.560000,13.240000
2,616172,<NA>,NaN,Santiago,Fullsabor,1,1,1,1,0,1,0,0,0.0,14.00,14.000000,NaN,NaN,NaN,NaN,0.000000,1.000000,0.000000,NaN,14.000000,0.000000,14.000000
3,452843,<NA>,NaN,Puerto varas,King Wins,9,1,9,9,0,0,1,0,0.0,90.00,10.000000,11.244444,21.244444,263.830000,24.333333,0.000000,0.000000,0.111111,263.830000,10.000000,0.000000,10.000000
4,629758,<NA>,NaN,Osorno,La Papa De Moka´s.,2,1,2,2,1,1,1,0,0.0,32.00,16.000000,1.070000,17.070000,250.650000,17.000000,0.500000,0.500000,0.500000,250.650000,16.000000,0.000000,16.000000
5,599640,<NA>,NaN,Copiapo,Panaderia Y Pizzeria Vincenzo Pezzuoli,44,1,44,44,0,3,39,0,0.0,453.01,10.295682,2.883864,13.179545,253.030000,16.160455,0.000000,0.068182,0.886364,253.030000,10.295682,0.000000,10.295682
6,542387,<NA>,NaN,Santiago,Rapaz Burgers - Tobalaba,4,1,4,4,4,4,1,0,0.0,149.00,37.250000,5.885000,43.135000,270.820000,46.250000,1.000000,1.000000,0.250000,270.820000,37.250000,0.000000,37.250000
7,573294,<NA>,NaN,Santiago,Satoru Sushi 794,8,1,8,8,0,1,6,0,0.0,128.00,16.000000,1.970000,17.970000,255.781667,17.125000,0.000000,0.125000,0.750000,255.781667,16.000000,0.000000,16.000000
8,597098,<NA>,NaN,Valdivia,Sushi Del Brujo.,20,1,20,20,0,2,20,0,0.0,353.98,17.699000,4.636000,22.336500,261.693000,23.100000,0.000000,0.100000,1.000000,261.693000,17.699000,0.000000,17.699000


## Funciones

### weighted_percentile()

In [18]:
def weighted_percentile(group, value_col, weight_col, percentile_val):
    values = group[value_col].to_numpy()
    weights = group[weight_col].to_numpy()

    # Filter out NaNs from values and weights, and weights that are zero or negative
    valid_mask = ~np.isnan(values) & ~np.isnan(weights) & (weights > 0)
    values = values[valid_mask]
    weights = weights[valid_mask]

    if len(values) == 0:
        return np.nan

    # Sort values and corresponding weights
    idx = np.argsort(values)
    sorted_values = values[idx]
    sorted_weights = weights[idx]

    # Calculate cumulative sum of weights
    cumulative_weights = np.cumsum(sorted_weights)
    total_weight = cumulative_weights[-1]

    if total_weight == 0: # This case should be rare after filtering, but for safety
        return np.nan

    # Find the index where the cumulative weight exceeds the threshold
    threshold_weight = total_weight * (percentile_val / 100.0)

    # Use searchsorted to find the index of the first value whose cumulative weight
    # is greater than or equal to the threshold_weight.
    interp_idx = np.searchsorted(cumulative_weights, threshold_weight, side='left')

    # Handle edge cases:
    if percentile_val == 0:
        return sorted_values[0]
    if percentile_val == 100:
        return sorted_values[-1]
    if interp_idx >= len(sorted_values): # Should only happen if percentile_val is 100
        return sorted_values[-1]

    return sorted_values[interp_idx]

median_cap()

In [19]:
def median_cap(
    df,
    value_col="ept_new",
    weight_col="total_orders",
    group_col="vendor_code",
    percentile=50,
    multiplier=1.2
):
    """
    Capea value_col a multiplier * weighted_percentile(group).

    Retorna una copia del dataframe con:
        - vendor_percentile
        - upper_limit
        - is_capped
        - value_col modificado
    """

    new_df = df.copy()

    percentile_df = (
        new_df
        .groupby(group_col)
        .apply(
            lambda x: weighted_percentile(
                x,
                value_col,
                weight_col,
                percentile
            ),
            include_groups=False
        )
        .reset_index(name="vendor_percentile")
    )

    new_df = new_df.merge(
        percentile_df,
        on=group_col,
        how="left"
    )

    new_df["upper_limit"] = (
        new_df["vendor_percentile"] * multiplier
    )

    new_df["is_capped"] = (
        new_df[value_col] > new_df["upper_limit"]
    )

    new_df[value_col] = np.where(
        new_df["is_capped"],
        new_df["upper_limit"],
        new_df[value_col]
    )

    print(f"Number of rows capped: {new_df['is_capped'].sum():,}")

    return new_df

### aplicar_factor_ept()

In [20]:
import numpy as np
import pandas as pd

def aplicar_factor_ept(ept_actuales, df_reduction):
    df_out = ept_actuales.copy()

    store_col = "vendor_code"
    ept_col = "ept"
    reduction_col = "ept_reduction_pct"

    # Asegurar mismo tipo para merge
    df_out[store_col] = df_out[store_col].astype(str)

    factors = (
        df_reduction[[store_col, reduction_col]]
        .copy()
    )

    factors[store_col] = factors[store_col].astype(str)

    # Si viene como 13 en vez de 0.13, lo transforma a 0.13
    factors[reduction_col] = np.where(
        factors[reduction_col] > 1,
        factors[reduction_col] / 100,
        factors[reduction_col]
    )

    # Una fila por vendor_code
    factors = (
        factors
        .drop_duplicates(subset=[store_col])
        .rename(columns={reduction_col: "ept_reduction_pct_applied"})
    )

    # Merge left para NO perder filas de ept_actuales
    df_out = df_out.merge(
        factors,
        on=store_col,
        how="left"
    )

    # Vendors sin factor quedan iguales. No se hace clip porque también
    # se permiten aumentos de EPT (porcentaje de reducción negativo).
    df_out["ept_reduction_pct_applied"] = (
        pd.to_numeric(
            df_out["ept_reduction_pct_applied"],
            errors="coerce"
        ).fillna(0)
    )

    # Aplicar reducción
    df_out["ept_new"] = (
        df_out[ept_col] * (1 - df_out["ept_reduction_pct_applied"])
    )

    df_out["ept_delta_min"] = df_out["ept_new"] - df_out[ept_col]

    print(f"Avg EPT actual: {df_out[ept_col].mean():.2f}")
    print(f"Avg EPT nuevo:  {df_out['ept_new'].mean():.2f}")
    print(f"Delta avg EPT:  {df_out['ept_new'].mean() - df_out[ept_col].mean():.2f}")

    return df_out

### exportar_template_ops()

In [21]:
import os
import pandas as pd
from datetime import datetime

def exportar_template_ops(ept_actuales_sim, max_num=900):
    df = ept_actuales_sim.copy()

    dest_folder = '/content/drive/MyDrive/lower ept y awt/results'
    os.makedirs(dest_folder, exist_ok=True)

    df['day_of_week_name_clean'] = df['day_of_week_name'].astype(str).str.upper().str.strip()

    day_map = {
        'MONDAY': 'MONDAY-MONDAY',
        'TUESDAY': 'TUESDAY-TUESDAY',
        'WEDNESDAY': 'WEDNESDAY-WEDNESDAY',
        'THURSDAY': 'THURSDAY-THURSDAY',
        'FRIDAY': 'FRIDAY-FRIDAY',
        'SATURDAY': 'SATURDAY-SATURDAY',
        'SUNDAY': 'SUNDAY-SUNDAY',
        'LUNES': 'MONDAY-MONDAY',
        'MARTES': 'TUESDAY-TUESDAY',
        'MIERCOLES': 'WEDNESDAY-WEDNESDAY',
        'MIÉRCOLES': 'WEDNESDAY-WEDNESDAY',
        'JUEVES': 'THURSDAY-THURSDAY',
        'VIERNES': 'FRIDAY-FRIDAY',
        'SABADO': 'SATURDAY-SATURDAY',
        'SÁBADO': 'SATURDAY-SATURDAY',
        'DOMINGO': 'SUNDAY-SUNDAY'
    }

    df['DAY-RANGE'] = df['day_of_week_name_clean'].map(day_map)

    df['time_block_clean'] = df['time_block'].astype(str).str.upper().str.strip()

    hour_map = {
        'LUNCH': ['12-14'],
        'DINNER': ['19-21'],
        'VALLE': ['0-11', '15-18', '22-23'],
        'OFF_PEAK': ['0-11', '15-18', '22-23'],
        'RESTO': ['0-11', '15-18', '22-23']
    }

    df['HOUR-RANGE'] = df['time_block_clean'].map(hour_map)
    df = df.explode('HOUR-RANGE')

    df_template = pd.DataFrame({
        'CODE': df['vendor_code'].astype(str),
        'DAY-RANGE': df['DAY-RANGE'].astype(str),
        # Fórmula de texto para que Google Sheets NO lo transforme a fecha
        'HOUR-RANGE': df['HOUR-RANGE'].astype(str),
        'PREPARATION-BUFFER': 2,
        'PREPARATION-TIME': df['ept_new'].astype(int),
        'STRATEGY': 'OPS_TEMPORARY'
    })

    # Ordenar para que las filas de cada CODE queden juntas
    df_template = df_template.sort_values(
        ['CODE', 'DAY-RANGE', 'HOUR-RANGE']
    ).reset_index(drop=True)

    # Validar que ningún CODE supere max_num filas por sí solo
    rows_by_code = df_template.groupby('CODE').size()

    codes_over_limit = rows_by_code[rows_by_code > max_num]

    if len(codes_over_limit) > 0:
        raise ValueError(
            f"Hay CODEs con más de max_num filas, imposible mantenerlos completos en un solo archivo: "
            f"{codes_over_limit.to_dict()}"
        )

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    files_created = []
    current_chunk = []
    current_rows = 0
    part = 1

    for code, df_code in df_template.groupby('CODE', sort=False):
        code_rows = len(df_code)

        if current_rows + code_rows > max_num:
            chunk_df = pd.concat(current_chunk, ignore_index=True)

            filename = f'template_ops_tiempos_{timestamp}_part_{part:02d}.csv'
            output_path = os.path.join(dest_folder, filename)

            chunk_df.to_csv(output_path, index=False)

            files_created.append(output_path)

            part += 1
            current_chunk = []
            current_rows = 0

        current_chunk.append(df_code)
        current_rows += code_rows

    # Guardar último chunk
    if current_chunk:
        chunk_df = pd.concat(current_chunk, ignore_index=True)

        filename = f'template_ops_tiempos_{timestamp}_part_{part:02d}.csv'
        output_path = os.path.join(dest_folder, filename)

        chunk_df.to_csv(output_path, index=False)

        files_created.append(output_path)

    print(f"Filas totales exportadas: {len(df_template):,}")
    print(f"Archivos creados: {len(files_created):,}")

    for path in files_created:
        print(path)

    return df_template

### rellenar_vendor_dow_block()

In [22]:
import numpy as np
import pandas as pd

def rellenar_vendor_dow_block(
    df,
    target_time_blocks=['lunch', 'dinner', 'valle']
):
    df = df.copy()

    vendor_col = 'vendor_code'
    dow_num_col = 'day_of_week_num'
    dow_name_col = 'day_of_week_name'
    block_col = 'time_block'
    weight_col = 'total_orders'

    # Explicitly convert day_of_week_num to numeric
    df[dow_num_col] = pd.to_numeric(df[dow_num_col], errors='coerce')

    days_df = pd.DataFrame({
        dow_num_col: [1, 2, 3, 4, 5, 6, 7],
        dow_name_col: ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
    })

    meta_cols = [
        'franchise_id', 'franchise_name', 'city_name',
        'franchise_orders', 'franchise_stores',
        'store_name', 'store_orders_total'
    ]

    count_cols = [
        'total_orders', 'slow_orders', 'non_seamless_orders'
    ]

    metric_cols = [
        'ept', 'awt', 'tt', 'ctp'
    ]

    meta_cols = [c for c in meta_cols if c in df.columns]
    count_cols = [c for c in count_cols if c in df.columns]
    metric_cols = [c for c in metric_cols if c in df.columns]

    for col in count_cols + metric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    def first_notna(s):
        s = s.dropna()
        return s.iloc[0] if len(s) else np.nan

    def wavg(g, col):
        mask = g[col].notna()

        if weight_col in g.columns:
            mask = mask & g[weight_col].notna() & (g[weight_col] > 0)

            if mask.sum() > 0:
                return np.average(g.loc[mask, col], weights=g.loc[mask, weight_col])

        return g[col].mean()

    def summarize(keys, suffix=None):
        rows = []

        for key_values, g in df.groupby(keys, dropna=False):
            if not isinstance(key_values, tuple):
                key_values = (key_values,)

            row = dict(zip(keys, key_values))

            for col in metric_cols:
                out_col = f'{col}_{suffix}' if suffix else col
                row[out_col] = wavg(g, col)

            rows.append(row)

        return pd.DataFrame(rows)

    # 1) Metadata única por vendor
    vendor_meta = (
        df[[vendor_col] + meta_cols]
        .groupby(vendor_col, as_index=False)
        .agg({c: first_notna for c in meta_cols})
    )

    # 2) Grilla completa vendor x dow x block
    all_blocks_df = pd.DataFrame({block_col: target_time_blocks})

    grid = (
        df[[vendor_col]]
        .drop_duplicates()
        .merge(days_df, how='cross')
        .merge(all_blocks_df, how='cross')
    )

    # 3) Data real agregada por vendor + dow + block
    real_rows = []
    group_keys = [vendor_col, dow_num_col, dow_name_col, block_col]

    # Define all possible columns for real_df including metrics and auxiliary flag
    real_df_cols = group_keys + count_cols + metric_cols + ['_is_real_row']

    # Only iterate if df is not empty to avoid issues with groupby on empty DataFrame
    if not df.empty:
        for key_values, g in df.groupby(group_keys, dropna=False):
            row = dict(zip(group_keys, key_values))

            for col in count_cols:
                row[col] = g[col].sum(min_count=1)

            for col in metric_cols:
                row[col] = wavg(g, col)

            row['_is_real_row'] = 1
            real_rows.append(row)

    # Ensure real_df has defined columns even if it's empty
    real_df = pd.DataFrame(real_rows, columns=real_df_cols)

    # 4) Promedio vendor + dow: para missing block
    vendor_dow_imp = summarize(
        [vendor_col, dow_num_col, dow_name_col],
        suffix='vendor_dow'
    )

    # 5) Promedio vendor + block: para missing dow completo
    vendor_block_imp = summarize(
        [vendor_col, block_col],
        suffix='vendor_block'
    )

    # 6) Promedio vendor completo: fallback final
    vendor_imp = summarize(
        [vendor_col],
        suffix='vendor'
    )

    # 7) Merge de todo
    out = (
        grid
        .merge(real_df, on=group_keys, how='left')
        .merge(vendor_meta, on=vendor_col, how='left')
        .merge(vendor_dow_imp, on=[vendor_col, dow_num_col, dow_name_col], how='left')
        .merge(vendor_block_imp, on=[vendor_col, block_col], how='left')
        .merge(vendor_imp, on=vendor_col, how='left')
    )

    # 8) Imputar métricas con prioridad:
    # real > vendor+dow > vendor+block > vendor
    for col in metric_cols:
        out[col] = (
            out[col]
            .fillna(out[f'{col}_vendor_dow'])
            .fillna(out[f'{col}_vendor_block'])
            .fillna(out[f'{col}_vendor'])
        )

    # 9) Filas artificiales no tienen orders reales
    for col in count_cols:
        out[col] = out[col].fillna(0)

    # 10) Flag de auditoría
    out['imputation_level'] = np.select(
        [
            out['_is_real_row'].eq(1),
            out[f'{metric_cols[0]}_vendor_dow'].notna(),
            out[f'{metric_cols[0]}_vendor_block'].notna(),
            out[f'{metric_cols[0]}_vendor'].notna()
        ],
        [
            'real',
            'vendor_dow_avg',
            'vendor_block_avg',
            'vendor_avg'
        ],
        default='no_data'
    )

    # 11) Limpiar auxiliares
    aux_cols = ['_is_real_row']

    for col in metric_cols:
        aux_cols += [
            f'{col}_vendor_dow',
            f'{col}_vendor_block',
            f'{col}_vendor'
        ]

    out = out.drop(columns=[c for c in aux_cols if c in out.columns])

    # 12) Orden final
    final_cols = [
        'franchise_id', 'franchise_name', 'city_name',
        'franchise_orders', 'franchise_stores',
        'vendor_code', 'store_name', 'store_orders_total',
        'day_of_week_num', 'day_of_week_name', 'time_block',
        'total_orders', 'slow_orders', 'non_seamless_orders',
        'ept', 'awt', 'tt', 'ctp',
        'imputation_level'
    ]

    final_cols = [c for c in final_cols if c in out.columns]

    out = (
        out[final_cols]
        .sort_values(
            ['franchise_name', 'store_name', 'day_of_week_num', 'time_block']
        )
        .reset_index(drop=True)
    )


    # Auditoría final
    n_vendors = out['vendor_code'].nunique()
    expected_rows = n_vendors * 7 * len(target_time_blocks)

    print(f'Vendors procesados: {n_vendors}')
    print(f'Filas generadas: {len(out)} de {expected_rows} esperadas')
    print(f'Filas con EPT NaN: {out["ept"].isna().sum()}')

    print('\nImputation levels:')
    print(out['imputation_level'].value_counts(dropna=False))

    vendors_no_data = out.loc[
        out['ept'].isna(),
        'vendor_code'
    ].drop_duplicates().tolist()

    if vendors_no_data:
        print(f'\nVendors sin EPT utilizable: {vendors_no_data}')
    else:
        print('\nTodos los vendors tienen EPT.')

    duplicates = out.duplicated(
        ['vendor_code', 'day_of_week_num', 'time_block']
    ).sum()

    print(f'Duplicados vendor+dow+block: {duplicates}')

    return out

# Calculo new EPTs

### calcular factor de EPT

In [23]:
# Normalizar IDs también en la tabla agregada.
df_grouped["vendor_code"] = normalize_id(df_grouped["vendor_code"])
df_grouped["franchise_id"] = normalize_id(df_grouped["franchise_id"])

rule_metadata = ["ept_new", "wave", "flag", "_input_row_number"]

# Las reglas de franquicia solo son filas sin vendor_code.
franchise_rules = (
    new_preps.loc[
        new_preps["adjustment_scope"].eq("franchise"),
        ["franchise_id"] + rule_metadata
    ]
    .drop_duplicates("franchise_id")
    .rename(columns={
        "ept_new": "ept_new_franchise",
        "wave": "wave_franchise",
        "flag": "flag_franchise",
        "_input_row_number": "input_row_number_franchise"
    })
)

# Toda fila con vendor_code es individual y tiene prioridad sobre la franquicia.
vendor_rules = (
    new_preps.loc[
        new_preps["adjustment_scope"].eq("vendor"),
        ["vendor_code"] + rule_metadata
    ]
    .drop_duplicates("vendor_code")
    .rename(columns={
        "ept_new": "ept_new_vendor",
        "wave": "wave_vendor",
        "flag": "flag_vendor",
        "_input_row_number": "input_row_number_vendor"
    })
)

df_reduction = (
    df_grouped
    .merge(franchise_rules, on="franchise_id", how="left")
    .merge(vendor_rules, on="vendor_code", how="left")
)

vendor_rule_applies = df_reduction["ept_new_vendor"].notna()
franchise_rule_applies = (
    ~vendor_rule_applies
    & df_reduction["ept_new_franchise"].notna()
)

df_reduction["ept_new"] = (
    df_reduction["ept_new_vendor"]
    .combine_first(df_reduction["ept_new_franchise"])
)
df_reduction["wave"] = df_reduction["wave_vendor"].where(
    vendor_rule_applies,
    df_reduction["wave_franchise"]
)
df_reduction["flag"] = df_reduction["flag_vendor"].where(
    vendor_rule_applies,
    df_reduction["flag_franchise"]
)
df_reduction["input_row_number"] = df_reduction[
    "input_row_number_vendor"
].where(
    vendor_rule_applies,
    df_reduction["input_row_number_franchise"]
)

df_reduction["adjustment_scope"] = np.select(
    [vendor_rule_applies, franchise_rule_applies],
    ["vendor", "franchise"],
    default="not_targeted"
)
df_reduction["source_franchise_id"] = df_reduction[
    "franchise_id"
].where(df_reduction["adjustment_scope"].eq("franchise"))

has_wave = (
    df_reduction["wave"].astype("string").str.strip().ne("").fillna(False)
)
df_reduction["adjustment_origin"] = np.select(
    [
        df_reduction["adjustment_scope"].eq("not_targeted"),
        has_wave
    ],
    ["not_targeted", "wave"],
    default="manual"
)

# Diferencia entre EPT actual promedio y EPT objetivo solicitado.
df_reduction["ept_diff"] = df_reduction["ept"] - df_reduction["ept_new"]
is_targeted = df_reduction["ept_new"].notna()

if reductions_only:
    df_reduction["apply_change"] = (
        is_targeted
        & (
            df_reduction["ept"].isna()
            | (df_reduction["ept_diff"] >= required_diff_mins)
        )
    )
else:
    df_reduction["apply_change"] = (
        is_targeted
        & (
            df_reduction["ept"].isna()
            | (df_reduction["ept_diff"].abs() >= required_diff_mins)
        )
    )

# Porcentaje que después aplica aplicar_factor_ept().
df_reduction["ept_reduction_pct"] = np.where(
    df_reduction["apply_change"],
    1 - (df_reduction["ept_new"] / df_reduction["ept"]),
    0
)

df_reduction[[
    "franchise_id",
    "vendor_code",
    "adjustment_origin",
    "adjustment_scope",
    "wave",
    "flag",
    "ept",
    "ept_new",
    "ept_diff",
    "apply_change",
    "ept_reduction_pct"
]].head()


/tmp/ipykernel_4417/505236277.py:51: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  .combine_first(df_reduction["ept_new_franchise"])


,franchise_id,vendor_code,adjustment_origin,adjustment_scope,wave,flag,ept,ept_new,ept_diff,apply_change,ept_reduction_pct
0,0016900002ZTfJUAA1,100004,manual,vendor,<NA>,Manual modification,12.120267,15,-2.879733,False,0.000000
1,<NA>,168677,manual,vendor,<NA>,Manual modification,12.680000,20,-7.320000,True,-0.577287
2,<NA>,616172,manual,vendor,<NA>,Manual modification,14.000000,25,-11.000000,True,-0.785714
3,<NA>,452843,manual,vendor,<NA>,Manual modification,10.000000,20,-10.000000,True,-1.000000
4,<NA>,629758,manual,vendor,<NA>,Manual modification,16.000000,20,-4.000000,True,-0.250000


### parchar dow-blocks faltantes con avg

In [24]:
# Procesar únicamente vendors resueltos por alguna regla vigente.
target_vendor_codes = set(
    df_reduction.loc[df_reduction["apply_change"]==True, "vendor_code"]
    .dropna()
)

ept_actuales_targeted = ept_actuales.loc[
    ept_actuales["vendor_code"].isin(target_vendor_codes)
].copy()

ept_actuales_mod_patched = rellenar_vendor_dow_block(ept_actuales_targeted)

Vendors procesados: 8
Filas generadas: 168 de 168 esperadas
Filas con EPT NaN: 0

Imputation levels:
imputation_level
real                57
vendor_block_avg    46
vendor_dow_avg      39
vendor_avg          26
Name: count, dtype: int64

Todos los vendors tienen EPT.
Duplicados vendor+dow+block: 0


### aplicar factor EPT

In [25]:
ept_actuales_new_ept = aplicar_factor_ept(
    ept_actuales_mod_patched,
    df_reduction
)

print(
    f"Stores a modificar: "
    f"{df_reduction.loc[df_reduction['apply_change'], 'vendor_code'].nunique()}"
)

print(
    f"De un total objetivo de: "
    f"{df_reduction.loc[df_reduction['ept_new'].notna(), 'vendor_code'].nunique()}"
)

origin_summary = (
    df_reduction.loc[
        df_reduction["ept_new"].notna(),
        ["vendor_code", "adjustment_origin", "adjustment_scope"]
    ]
    .groupby(
        ["adjustment_origin", "adjustment_scope"],
        dropna=False,
        as_index=False
    )
    .agg(vendors=("vendor_code", "nunique"))
)

print("\nOrigen y alcance del EPT objetivo:")
print(origin_summary.to_string(index=False))


Avg EPT actual: 16.78
Avg EPT nuevo:  22.30
Delta avg EPT:  5.52
Stores a modificar: 8
De un total objetivo de: 9

Origen y alcance del EPT objetivo:
adjustment_origin adjustment_scope  vendors
           manual           vendor        9


In [26]:
# Vendors que tienen al menos un EPT histórico numérico válido.
ept_history_numeric = pd.to_numeric(ept_actuales["ept"], errors="coerce")
vendors_with_history = set(
    ept_actuales.loc[ept_history_numeric.notna(), "vendor_code"].dropna()
)

# Targets ya expandidos por la query: incluyen los vendors de cada franquicia.
target_meta_cols = [
    "vendor_code", "franchise_id", "franchise_name",
    "city_name", "store_name", "ept_new",
    "adjustment_origin", "adjustment_scope", "source_franchise_id",
    "wave", "flag", "input_row_number", "apply_change"
]
target_meta_cols = [
    column for column in target_meta_cols
    if column in df_reduction.columns
]

resolved_targets = (
    df_reduction.loc[df_reduction["ept_new"].notna(), target_meta_cols]
    .drop_duplicates("vendor_code")
    .copy()
)

# Fallback para un vendor explícito que no apareció en la query.
unresolved_explicit = vendor_rules.rename(columns={
    "ept_new_vendor": "ept_new",
    "wave_vendor": "wave",
    "flag_vendor": "flag",
    "input_row_number_vendor": "input_row_number"
}).copy()

for column in [
    "franchise_id", "franchise_name", "city_name", "store_name"
]:
    unresolved_explicit[column] = pd.NA

unresolved_explicit["adjustment_scope"] = "vendor"
unresolved_explicit["source_franchise_id"] = pd.NA
unresolved_explicit["adjustment_origin"] = np.where(
    unresolved_explicit["wave"].astype("string").str.strip().ne("").fillna(False),
    "wave",
    "manual"
)
unresolved_explicit["apply_change"] = True
unresolved_explicit = unresolved_explicit.reindex(columns=target_meta_cols)

all_targets = (
    pd.concat(
        [resolved_targets, unresolved_explicit],
        ignore_index=True,
        sort=False
    )
    .drop_duplicates("vendor_code", keep="first")
)

missing_vendors = all_targets.loc[
    ~all_targets["vendor_code"].isin(vendors_with_history)
].copy()

# Quitar sus filas actuales con EPT NaN para evitar duplicados.
ept_actuales_new_ept = ept_actuales_new_ept.loc[
    ~ept_actuales_new_ept["vendor_code"].astype(str).isin(
        missing_vendors["vendor_code"].astype(str)
    )
].copy()

# Crear las 21 combinaciones: 7 días x 3 bloques.
days = pd.DataFrame({
    "day_of_week_num": range(1, 8),
    "day_of_week_name": [
        "Sunday", "Monday", "Tuesday", "Wednesday",
        "Thursday", "Friday", "Saturday"
    ]
})

blocks = pd.DataFrame({
    "time_block": ["lunch", "dinner", "valle"]
})

missing_rows = (
    missing_vendors
    .merge(days, how="cross")
    .merge(blocks, how="cross")
)

# Sin historia, usar directamente el EPT objetivo solicitado.
missing_rows["ept"] = missing_rows["ept_new"]
missing_rows["ept_delta_min"] = 0
missing_rows["total_orders"] = 1
missing_rows["imputation_level"] = "new_preps_no_history"

ept_actuales_new_ept = pd.concat(
    [ept_actuales_new_ept, missing_rows],
    ignore_index=True,
    sort=False
)

# Los vendors explícitos sin match también deben llegar al template TES.
target_vendor_codes.update(
    missing_vendors["vendor_code"].dropna().astype(str).tolist()
)

print(
    f"Vendors sin historia agregados: "
    f"{missing_vendors['vendor_code'].nunique()}"
)


Vendors sin historia agregados: 0


/tmp/ipykernel_4417/2971403817.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(


### (check)

In [27]:
# checkear random vendors
random_vendor_code = ept_actuales_new_ept['vendor_code'].sample(1).iloc[0]
ept_actuales_new_ept[ept_actuales_new_ept['vendor_code'] == random_vendor_code][[
    'franchise_name',
    'vendor_code',
    'day_of_week_name',
    'time_block',
    'ept',
    'ept_new',
    'ept_delta_min',
    # 'ept_reduction_pct'
]].round(2)

,franchise_name,vendor_code,day_of_week_name,time_block,ept,ept_new,ept_delta_min
0,NaN,168677,Sunday,dinner,13.0,20.50,7.50
1,NaN,168677,Sunday,lunch,14.5,22.87,8.37
2,NaN,168677,Sunday,valle,13.0,20.50,7.50
3,NaN,168677,Monday,dinner,13.0,20.50,7.50
4,NaN,168677,Monday,lunch,13.0,20.50,7.50
5,NaN,168677,Monday,valle,13.0,20.50,7.50
6,NaN,168677,Tuesday,dinner,13.0,20.50,7.50
7,NaN,168677,Tuesday,lunch,13.0,20.50,7.50
8,NaN,168677,Tuesday,valle,13.0,20.50,7.50
9,NaN,168677,Wednesday,dinner,13.0,20.50,7.50


### capping

In [28]:
random_vendor_code = ept_actuales_new_ept['vendor_code'].sample(1).iloc[0]

ept_actuales_new_ept_capped = median_cap(
    ept_actuales_new_ept,
    multiplier=0.8
)
ept_actuales_new_ept_capped[ept_actuales_new_ept_capped['vendor_code'] == random_vendor_code][[
    'franchise_name',
    'vendor_code',
    'day_of_week_name',
    'time_block',
    'ept',
    'ept_new',
    'ept_delta_min',
    'upper_limit',
    'is_capped'
    # 'ept_reduction_pct'
]].round(2)

# ept_actuales_new_ept_capped[ept_actuales_new_ept_capped['vendor_code'] == random_vendor_code]

Number of rows capped: 168


,franchise_name,vendor_code,day_of_week_name,time_block,ept,ept_new,ept_delta_min,upper_limit,is_capped
21,NaN,616172,Sunday,dinner,14.0,20.0,11.0,20.0,True
22,NaN,616172,Sunday,lunch,14.0,20.0,11.0,20.0,True
23,NaN,616172,Sunday,valle,14.0,20.0,11.0,20.0,True
24,NaN,616172,Monday,dinner,14.0,20.0,11.0,20.0,True
25,NaN,616172,Monday,lunch,14.0,20.0,11.0,20.0,True
26,NaN,616172,Monday,valle,14.0,20.0,11.0,20.0,True
27,NaN,616172,Tuesday,dinner,14.0,20.0,11.0,20.0,True
28,NaN,616172,Tuesday,lunch,14.0,20.0,11.0,20.0,True
29,NaN,616172,Tuesday,valle,14.0,20.0,11.0,20.0,True
30,NaN,616172,Wednesday,dinner,14.0,20.0,11.0,20.0,True


### formato TES y exportación

In [29]:
# ept_actuales_mod_TES = exportar_template_ops(df_manual_modified)
ept_actuales_mod_TES = exportar_template_ops(ept_actuales_new_ept_capped[ept_actuales_new_ept_capped["vendor_code"].isin(target_vendor_codes)], max_num = 99999) # todo a una pura hoja
ept_actuales
# resultados en: https://drive.google.com/drive/u/0/folders/1eRW_EDJ0hy687Ns1RZddRHJoXTTDFzmA

Filas totales exportadas: 280
Archivos creados: 1
/content/drive/MyDrive/lower ept y awt/results/template_ops_tiempos_20260904_212732_part_01.csv


,franchise_id,franchise_name,city_name,vertical_type,franchise_orders,franchise_stores,franchise_orders_with_fir,franchise_fir,vendor_code,store_name,store_orders_total,store_orders_with_fir,store_fir,day_of_week_num,day_of_week_name,time_block,total_orders,slow_orders,non_seamless_orders,hd_total_orders,hd_total_minutes_added,started_triggers,avg_trigger_duration_min,orders_with_fir,fir_ratio,ept,awt,tt,ctp,fir
0,<NA>,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,2,Monday,dinner,1,0,0,0,0.0,0,NaN,1,1.0,13.00,5.10,18.10,19.00,256.05
1,<NA>,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,2,Monday,valle,2,0,0,0,0.0,0,NaN,1,0.5,13.00,8.23,21.23,24.00,258.25
2,<NA>,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,3,Tuesday,lunch,1,0,0,0,0.0,0,NaN,1,1.0,13.00,NaN,NaN,24.00,263.47
3,<NA>,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,3,Tuesday,dinner,3,1,1,0,0.0,0,NaN,3,1.0,13.00,16.72,29.72,37.67,276.97
4,<NA>,None,Valdivia,restaurants,25,1,14,0.56,168677,Cocavi Delivery,25,14,0.56,3,Tuesday,valle,1,0,0,1,8.0,0,NaN,1,1.0,13.00,15.08,28.08,29.00,267.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,0016900002ZTfJUAA1,Montalbano,Calama,restaurants,75,1,72,0.96,100004,Schopdog Calama,75,72,0.96,6,Friday,dinner,1,0,0,0,0.0,0,NaN,1,1.0,15.00,NaN,NaN,25.00,250.22
73,0016900002ZTfJUAA1,Montalbano,Calama,restaurants,75,1,72,0.96,100004,Schopdog Calama,75,72,0.96,6,Friday,valle,3,0,0,0,0.0,0,NaN,3,1.0,11.33,2.89,14.23,14.33,250.37
74,0016900002ZTfJUAA1,Montalbano,Calama,restaurants,75,1,72,0.96,100004,Schopdog Calama,75,72,0.96,7,Saturday,lunch,10,1,2,4,32.0,0,NaN,10,1.0,16.60,3.91,20.51,24.90,253.90
75,0016900002ZTfJUAA1,Montalbano,Calama,restaurants,75,1,72,0.96,100004,Schopdog Calama,75,72,0.96,7,Saturday,dinner,1,0,0,0,0.0,1,1200.00,0,0.0,13.00,NaN,NaN,27.00,NaN


### auditoría y tabla estandarizada `results_export`

Convención de signos del output:

* `ept_change_min = ept_new - ept_old`.
* `ept_change_pct = ept_new / ept_old - 1`.
* Una reducción queda negativa y un alza queda positiva en ambas columnas.

`results_export` funciona como staging de la ejecución actual y se reemplaza en cada corrida. Para el historial consolidado en el otro spreadsheet, el nombre recomendado es `ept_adjustment_history`.


In [30]:
import os
from datetime import datetime
import numpy as np
import pandas as pd

# Este bloque debe ejecutarse después de crear ept_actuales_new_ept_capped.
summary_base = ept_actuales_new_ept_capped.copy()

numeric_cols = [
    "ept", "ept_new", "awt", "total_orders",
    "slow_orders", "non_seamless_orders"
]
for col in numeric_cols:
    if col not in summary_base.columns:
        summary_base[col] = np.nan
    summary_base[col] = pd.to_numeric(summary_base[col], errors="coerce")

# Las filas creadas para vendors sin historia traen total_orders=1 solo para
# permitir generar la grilla TES. No deben contarse como órdenes reales.
summary_base["has_real_history"] = (
    summary_base["ept"].notna()
    & summary_base["total_orders"].gt(0)
    & ~summary_base["imputation_level"].eq("new_preps_no_history")
)
summary_base["orders_for_summary"] = np.where(
    summary_base["has_real_history"],
    summary_base["total_orders"],
    0
)

# Es el valor que realmente se manda en PREPARATION-TIME, porque el
# exportador TES usa astype(int). Para EPT positivos equivale a truncar.
summary_base["ept_new_tes"] = np.trunc(summary_base["ept_new"])
summary_base["old_ept_minutes"] = (
    summary_base["ept"] * summary_base["orders_for_summary"]
)
summary_base["new_ept_minutes"] = (
    summary_base["ept_new"] * summary_base["orders_for_summary"]
)
summary_base["new_ept_tes_minutes"] = (
    summary_base["ept_new_tes"] * summary_base["orders_for_summary"]
)
summary_base["awt_minutes"] = (
    summary_base["awt"] * summary_base["orders_for_summary"]
)

def first_notna(series):
    values = series.dropna()
    return values.iloc[0] if len(values) else np.nan

# Metadata de la instrucción ganadora: una fila por vendor.
rule_cols = [
    "vendor_code", "ept_new", "adjustment_origin", "adjustment_scope",
    "source_franchise_id", "wave", "flag", "input_row_number",
    "apply_change"
]
rule_cols = [column for column in rule_cols if column in df_reduction.columns]
rule_meta_resolved = df_reduction[rule_cols].drop_duplicates("vendor_code")

# Los vendor_code explícitos sin historia no aparecen en df_reduction, pero
# sí se agregaron a la grilla final. Incorporarlos con la misma metadata.
missing_rule_cols = [
    "vendor_code", "ept_new", "adjustment_origin", "adjustment_scope",
    "source_franchise_id", "wave", "flag", "input_row_number",
    "apply_change"
]
missing_rule_cols = [
    column for column in missing_rule_cols
    if column in missing_vendors.columns
]
rule_meta_missing = missing_vendors[missing_rule_cols].copy()
rule_meta_missing["apply_change"] = True

rule_meta = (
    pd.concat([rule_meta_resolved, rule_meta_missing], ignore_index=True)
    .drop_duplicates("vendor_code", keep="first")
    .rename(columns={"ept_new": "ept_target_requested"})
)

vendor_rows = []
for vendor_code, g in summary_base.groupby("vendor_code", dropna=False):
    orders = g["orders_for_summary"].sum()

    if orders > 0:
        ept_old = g["old_ept_minutes"].sum() / orders
        ept_new = g["new_ept_minutes"].sum() / orders
        ept_new_tes = g["new_ept_tes_minutes"].sum() / orders
        awt_old = g["awt_minutes"].sum(min_count=1) / orders
        slow_orders = g.loc[g["has_real_history"], "slow_orders"].sum(min_count=1)
        non_seamless_orders = g.loc[
            g["has_real_history"], "non_seamless_orders"
        ].sum(min_count=1)
    else:
        ept_old = np.nan
        ept_new = g["ept_new"].mean()
        ept_new_tes = g["ept_new_tes"].mean()
        awt_old = np.nan
        slow_orders = np.nan
        non_seamless_orders = np.nan

    reduction_min = ept_old - ept_new if pd.notna(ept_old) else np.nan
    reduction_pct = (
        reduction_min / ept_old
        if pd.notna(ept_old) and ept_old != 0
        else np.nan
    )
    reduction_tes_min = (
        ept_old - ept_new_tes if pd.notna(ept_old) else np.nan
    )

    vendor_rows.append({
        "franchise_id": first_notna(g["franchise_id"]),
        "franchise_name": first_notna(g["franchise_name"]),
        "vendor_code": vendor_code,
        "store_name": first_notna(g["store_name"]),
        "city_name": first_notna(g["city_name"]),
        "has_history": orders > 0,
        "orders_7d": orders,
        "ept_old": ept_old,
        "ept_new": ept_new,
        "reduction_min": reduction_min,
        "reduction_pct": reduction_pct,
        "ept_new_tes": ept_new_tes,
        "reduction_tes_min": reduction_tes_min,
        "estimated_minutes_reduced_7d": (
            reduction_tes_min * orders
            if pd.notna(reduction_tes_min)
            else np.nan
        ),
        "awt_old": awt_old,
        "slow_share": (
            slow_orders / orders if orders > 0 and pd.notna(slow_orders) else np.nan
        ),
        "non_seamless_share": (
            non_seamless_orders / orders
            if orders > 0 and pd.notna(non_seamless_orders)
            else np.nan
        ),
        "imputed_blocks": int((g["imputation_level"] != "real").sum()),
        "capped_blocks": int(g["is_capped"].fillna(False).sum())
    })

vendor_summary = pd.DataFrame(vendor_rows).merge(
    rule_meta, on="vendor_code", how="left"
)
vendor_summary["apply_change"] = vendor_summary["apply_change"].fillna(False)
vendor_summary["change_type"] = np.select(
    [
        ~vendor_summary["has_history"],
        vendor_summary["apply_change"] & vendor_summary["reduction_tes_min"].gt(0),
        vendor_summary["apply_change"] & vendor_summary["reduction_tes_min"].lt(0)
    ],
    ["no_history", "reduction", "increase"],
    default="unchanged"
)

# Una tienda sin franchise_id constituye su propio grupo; así no se mezclan
# todos los independientes bajo un franchise NULL.
vendor_summary["franchise_group_id"] = np.where(
    vendor_summary["franchise_id"].notna(),
    vendor_summary["franchise_id"].astype("string"),
    "VENDOR_" + vendor_summary["vendor_code"].astype("string")
)
vendor_summary["franchise_group_name"] = vendor_summary["franchise_name"].fillna(
    "Independent / " + vendor_summary["store_name"].fillna(
        vendor_summary["vendor_code"].astype("string")
    )
)

franchise_rows = []
for group_id, g in vendor_summary.groupby("franchise_group_id", dropna=False):
    history = g["has_history"] & g["orders_7d"].gt(0)
    orders = g.loc[history, "orders_7d"].sum()

    if orders > 0:
        ept_old = np.average(
            g.loc[history, "ept_old"],
            weights=g.loc[history, "orders_7d"]
        )
        ept_new = np.average(
            g.loc[history, "ept_new"],
            weights=g.loc[history, "orders_7d"]
        )
        ept_new_tes = np.average(
            g.loc[history, "ept_new_tes"],
            weights=g.loc[history, "orders_7d"]
        )
        awt_old = np.average(
            g.loc[history & g["awt_old"].notna(), "awt_old"],
            weights=g.loc[history & g["awt_old"].notna(), "orders_7d"]
        ) if (history & g["awt_old"].notna()).any() else np.nan
    else:
        ept_old = np.nan
        ept_new = g["ept_new"].mean()
        ept_new_tes = g["ept_new_tes"].mean()
        awt_old = np.nan

    reduction_min = ept_old - ept_new if pd.notna(ept_old) else np.nan
    reduction_tes_min = (
        ept_old - ept_new_tes if pd.notna(ept_old) else np.nan
    )

    franchise_rows.append({
        "franchise_group_id": group_id,
        "franchise_id": first_notna(g["franchise_id"]),
        "franchise_name": first_notna(g["franchise_group_name"]),
        "vendors_exported": g["vendor_code"].nunique(),
        "vendors_affected": g.loc[g["apply_change"], "vendor_code"].nunique(),
        "vendors_reduced": g.loc[g["change_type"].eq("reduction"), "vendor_code"].nunique(),
        "vendors_increased": g.loc[g["change_type"].eq("increase"), "vendor_code"].nunique(),
        "vendors_unchanged": g.loc[g["change_type"].eq("unchanged"), "vendor_code"].nunique(),
        "vendors_without_history": g.loc[~g["has_history"], "vendor_code"].nunique(),
        "orders_7d": orders,
        "ept_old": ept_old,
        "ept_new": ept_new,
        "reduction_min": reduction_min,
        "reduction_pct": (
            reduction_min / ept_old
            if pd.notna(ept_old) and ept_old != 0
            else np.nan
        ),
        "ept_new_tes": ept_new_tes,
        "reduction_tes_min": reduction_tes_min,
        "estimated_minutes_reduced_7d": g["estimated_minutes_reduced_7d"].sum(min_count=1),
        "awt_old": awt_old,
        "slow_share": (
            np.average(
                g.loc[history & g["slow_share"].notna(), "slow_share"],
                weights=g.loc[history & g["slow_share"].notna(), "orders_7d"]
            ) if (history & g["slow_share"].notna()).any() else np.nan
        ),
        "non_seamless_share": (
            np.average(
                g.loc[history & g["non_seamless_share"].notna(), "non_seamless_share"],
                weights=g.loc[history & g["non_seamless_share"].notna(), "orders_7d"]
            ) if (history & g["non_seamless_share"].notna()).any() else np.nan
        )
    })

franchise_summary = pd.DataFrame(franchise_rows)

# Redondear solo para la salida; los cálculos anteriores mantienen precisión.
round_cols = [
    "ept_old", "ept_new", "reduction_min", "reduction_pct",
    "ept_new_tes", "reduction_tes_min",
    "estimated_minutes_reduced_7d", "awt_old",
    "slow_share", "non_seamless_share"
]
for df_out in [vendor_summary, franchise_summary]:
    cols = [c for c in round_cols if c in df_out.columns]
    df_out[cols] = df_out[cols].round(4)

vendor_summary = vendor_summary.sort_values(
    ["franchise_group_name", "store_name", "vendor_code"]
).reset_index(drop=True)
franchise_summary = franchise_summary.sort_values(
    ["franchise_name", "franchise_group_id"]
).reset_index(drop=True)

dest_folder = "/content/drive/MyDrive/lower ept y awt/results"
os.makedirs(dest_folder, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
vendor_path = os.path.join(
    dest_folder, f"summary_reducciones_vendor_{timestamp}.csv"
)
franchise_path = os.path.join(
    dest_folder, f"summary_reducciones_franchise_{timestamp}.csv"
)

vendor_summary.to_csv(vendor_path, index=False)
franchise_summary.to_csv(franchise_path, index=False)

print(f"CSV vendor: {vendor_path}")
print(f"CSV franchise: {franchise_path}")
print(
    f"Vendors exportados: {vendor_summary['vendor_code'].nunique():,} | "
    f"Vendors afectados: {vendor_summary['apply_change'].sum():,} | "
    f"Sin historia: {(~vendor_summary['has_history']).sum():,}"
)

display(franchise_summary.head(20))
display(vendor_summary.head(20))

CSV vendor: /content/drive/MyDrive/lower ept y awt/results/summary_reducciones_vendor_20260904_212735.csv
CSV franchise: /content/drive/MyDrive/lower ept y awt/results/summary_reducciones_franchise_20260904_212735.csv
Vendors exportados: 8 | Vendors afectados: 8 | Sin historia: 0


,franchise_group_id,franchise_id,franchise_name,vendors_exported,vendors_affected,vendors_reduced,vendors_increased,vendors_unchanged,vendors_without_history,orders_7d,ept_old,ept_new,reduction_min,reduction_pct,ept_new_tes,reduction_tes_min,estimated_minutes_reduced_7d,awt_old,slow_share,non_seamless_share
0,VENDOR_168677,NaN,Independent / Cocavi Delivery,1,1,0,1,0,0,25.0,13.2400,16.4038,-3.1638,-0.2390,16.0,-2.7600,-69.00,14.4714,0.16,0.2800
1,VENDOR_616172,NaN,Independent / Fullsabor,1,1,0,1,0,0,1.0,14.0000,20.0000,-6.0000,-0.4286,20.0,-6.0000,-6.00,NaN,0.00,1.0000
2,VENDOR_452843,NaN,Independent / King Wins,1,1,0,1,0,0,9.0,10.0000,16.0000,-6.0000,-0.6000,16.0,-6.0000,-54.00,11.2444,0.00,0.0000
3,VENDOR_629758,NaN,Independent / La Papa De Moka´s.,1,1,0,0,1,0,2.0,16.0000,16.0000,0.0000,0.0000,16.0,0.0000,0.00,1.0700,0.50,0.5000
4,VENDOR_599640,NaN,Independent / Panaderia Y Pizzeria Vincenzo Pe...,1,1,0,1,0,0,44.0,10.2957,11.8185,-1.5229,-0.1479,11.0,-0.7043,-30.99,2.8839,0.00,0.0682
5,VENDOR_542387,NaN,Independent / Rapaz Burgers - Tobalaba,1,1,1,0,0,0,4.0,37.2500,23.5168,13.7332,0.3687,23.0,14.2500,57.00,5.8850,1.00,1.0000
6,VENDOR_573294,NaN,Independent / Satoru Sushi 794,1,1,0,1,0,0,8.0,16.0000,20.0000,-4.0000,-0.2500,20.0,-4.0000,-32.00,1.9700,0.00,0.1250
7,VENDOR_597098,NaN,Independent / Sushi Del Brujo.,1,1,1,0,0,0,20.0,17.6990,17.6733,0.0257,0.0015,17.0,0.6990,13.98,4.6360,0.00,0.1000


,franchise_id,franchise_name,vendor_code,store_name,city_name,has_history,orders_7d,ept_old,ept_new,reduction_min,reduction_pct,ept_new_tes,reduction_tes_min,estimated_minutes_reduced_7d,awt_old,slow_share,non_seamless_share,imputed_blocks,capped_blocks,ept_target_requested,adjustment_origin,adjustment_scope,source_franchise_id,wave,flag,input_row_number,apply_change,change_type,franchise_group_id,franchise_group_name
0,NaN,NaN,168677,Cocavi Delivery,Valdivia,True,25.0,13.2400,16.4038,-3.1638,-0.2390,16.0,-2.7600,-69.00,14.4714,0.16,0.2800,7,21,20,manual,vendor,NaN,<NA>,Manual modification,7,True,increase,VENDOR_168677,Independent / Cocavi Delivery
1,NaN,NaN,616172,Fullsabor,Santiago,True,1.0,14.0000,20.0000,-6.0000,-0.4286,20.0,-6.0000,-6.00,NaN,0.00,1.0000,20,21,25,manual,vendor,NaN,<NA>,Manual modification,2,True,increase,VENDOR_616172,Independent / Fullsabor
2,NaN,NaN,452843,King Wins,Puerto varas,True,9.0,10.0000,16.0000,-6.0000,-0.6000,16.0,-6.0000,-54.00,11.2444,0.00,0.0000,15,21,20,manual,vendor,NaN,<NA>,Manual modification,10,True,increase,VENDOR_452843,Independent / King Wins
3,NaN,NaN,629758,La Papa De Moka´s.,Osorno,True,2.0,16.0000,16.0000,0.0000,0.0000,16.0,0.0000,0.00,1.0700,0.50,0.5000,19,21,20,manual,vendor,NaN,<NA>,Manual modification,9,True,unchanged,VENDOR_629758,Independent / La Papa De Moka´s.
4,NaN,NaN,599640,Panaderia Y Pizzeria Vincenzo Pezzuoli,Copiapo,True,44.0,10.2957,11.8185,-1.5229,-0.1479,11.0,-0.7043,-30.99,2.8839,0.00,0.0682,7,21,15,manual,vendor,NaN,<NA>,Manual modification,8,True,increase,VENDOR_599640,Independent / Panaderia Y Pizzeria Vincenzo Pe...
5,NaN,NaN,542387,Rapaz Burgers - Tobalaba,Santiago,True,4.0,37.2500,23.5168,13.7332,0.3687,23.0,14.2500,57.00,5.8850,1.00,1.0000,18,21,30,manual,vendor,NaN,<NA>,Manual modification,4,True,reduction,VENDOR_542387,Independent / Rapaz Burgers - Tobalaba
6,NaN,NaN,573294,Satoru Sushi 794,Santiago,True,8.0,16.0000,20.0000,-4.0000,-0.2500,20.0,-4.0000,-32.00,1.9700,0.00,0.1250,15,21,25,manual,vendor,NaN,<NA>,Manual modification,3,True,increase,VENDOR_573294,Independent / Satoru Sushi 794
7,NaN,NaN,597098,Sushi Del Brujo.,Valdivia,True,20.0,17.6990,17.6733,0.0257,0.0015,17.0,0.6990,13.98,4.6360,0.00,0.1000,10,21,23,manual,vendor,NaN,<NA>,Manual modification,6,True,reduction,VENDOR_597098,Independent / Sushi Del Brujo.


In [31]:
# ============================================================
# ESCRIBIR results_export Y MARCAR to_adjust_now
# ============================================================

import numpy as np
import pandas as pd
import gspread

# Una fila por vendor efectivamente incluido en el template TES.
export_audit = vendor_summary.loc[
    vendor_summary["apply_change"].fillna(False)
].copy()

# ept_new conserva el objetivo solicitado en to_adjust_now.
export_audit["ept_new"] = export_audit[
    "ept_target_requested"
].combine_first(export_audit["ept_new"])

# Convención: nuevo - antiguo. Reducciones negativas; alzas positivas.
export_audit["ept_change_min"] = (
    export_audit["ept_new"] - export_audit["ept_old"]
)
export_audit["ept_change_pct"] = np.where(
    export_audit["ept_old"].notna()
    & export_audit["ept_old"].ne(0),
    (export_audit["ept_new"] / export_audit["ept_old"]) - 1,
    np.nan
)
export_audit["executed_at"] = executed_at

# Esquema final mínimo para el paste y el historial consolidado.
result_columns = [
    "executed_at",
    "wave",
    "flag",
    "adjustment_scope",
    "franchise_id",
    "franchise_name",
    "vendor_code",
    "store_name",
    "ept_old",
    "ept_change_pct",
    "ept_new",
    "ept_change_min"
]

for column in result_columns:
    if column not in export_audit.columns:
        export_audit[column] = pd.NA

results_export = export_audit[result_columns].copy()

for column in ["ept_old", "ept_new", "ept_change_min"]:
    results_export[column] = pd.to_numeric(
        results_export[column], errors="coerce"
    ).round(1)

results_export["ept_change_pct"] = pd.to_numeric(
    results_export["ept_change_pct"], errors="coerce"
).round(4)

results_export = results_export.sort_values(
    ["wave", "franchise_name", "store_name", "vendor_code"],
    na_position="last"
).reset_index(drop=True)


def clean_sheet_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    return value


result_values = [result_columns] + [
    [clean_sheet_value(value) for value in row]
    for row in results_export.to_numpy()
]

try:
    results_worksheet = spreadsheet.worksheet(RESULTS_SHEET_NAME)
except gspread.WorksheetNotFound:
    results_worksheet = spreadsheet.add_worksheet(
        title=RESULTS_SHEET_NAME,
        rows=max(len(result_values), 2),
        cols=len(result_columns)
    )

results_worksheet.resize(
    rows=max(len(result_values), 2),
    cols=len(result_columns)
)
results_worksheet.clear()

result_end_cell = gspread.utils.rowcol_to_a1(
    len(result_values),
    len(result_columns)
)
results_worksheet.update(
    range_name=f"A1:{result_end_cell}",
    values=result_values,
    value_input_option="RAW"
)

# Formato visual; los datos ya quedan escritos aunque falle el formato.
try:
    results_worksheet.freeze(rows=1)
    last_column_letter = gspread.utils.rowcol_to_a1(
        1, len(result_columns)
    ).rstrip("1")
    results_worksheet.format(
        f"A1:{last_column_letter}1",
        {
            "backgroundColor": {"red": 0.12, "green": 0.28, "blue": 0.47},
            "textFormat": {
                "bold": True,
                "foregroundColor": {"red": 1, "green": 1, "blue": 1}
            }
        }
    )

    for column in ["ept_old", "ept_new", "ept_change_min"]:
        column_number = result_columns.index(column) + 1
        column_letter = gspread.utils.rowcol_to_a1(
            1, column_number
        ).rstrip("1")
        results_worksheet.format(
            f"{column_letter}2:{column_letter}{max(len(result_values), 2)}",
            {"numberFormat": {"type": "NUMBER", "pattern": "0.0"}}
        )

    change_pct_column_number = result_columns.index("ept_change_pct") + 1
    change_pct_column_letter = gspread.utils.rowcol_to_a1(
        1, change_pct_column_number
    ).rstrip("1")
    results_worksheet.format(
        f"{change_pct_column_letter}2:{change_pct_column_letter}{max(len(result_values), 2)}",
        {"numberFormat": {"type": "PERCENT", "pattern": "0%"}}
    )
except Exception as formatting_error:
    print(f"Advertencia de formato en results_export: {formatting_error}")

# Solo después de escribir results_export se marca el input como ejecutado.
if "last_executed_at" in input_headers_original:
    last_executed_at_col = input_headers_original.index("last_executed_at") + 1
    last_executed_values = [
        [row[last_executed_at_col - 1] if len(row) >= last_executed_at_col else ""]
        for row in input_values
    ]
    last_executed_values[0] = ["last_executed_at"]
else:
    last_executed_at_col = len(input_headers_original) + 1
    if last_executed_at_col > input_worksheet.col_count:
        input_worksheet.resize(
            rows=max(input_worksheet.row_count, len(input_values), 2),
            cols=last_executed_at_col
        )
    last_executed_values = [["last_executed_at"]] + [
        [""] for _ in input_values[1:]
    ]

for sheet_row_number in input_rows_to_mark:
    while len(last_executed_values) < sheet_row_number:
        last_executed_values.append([""])
    last_executed_values[sheet_row_number - 1] = [executed_at]

last_executed_at_column_letter = gspread.utils.rowcol_to_a1(
    1, last_executed_at_col
).rstrip("1")
input_worksheet.update(
    range_name=(
        f"{last_executed_at_column_letter}1:"
        f"{last_executed_at_column_letter}{len(last_executed_values)}"
    ),
    values=last_executed_values,
    value_input_option="RAW"
)

print(
    f"Hoja '{RESULTS_SHEET_NAME}' reemplazada con "
    f"{len(results_export):,} vendors exportados."
)
print(
    f"'{INPUT_SHEET_NAME}' se mantuvo intacta; "
    f"last_executed_at actualizado en {len(input_rows_to_mark):,} filas."
)
print("Origen del ajuste:")
display(
    export_audit.groupby(
        ["adjustment_origin", "adjustment_scope"],
        dropna=False
    ).size().rename("vendors").reset_index()
)
display(results_export.head(20))


/tmp/ipykernel_4417/2191539539.py:17: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  ].combine_first(export_audit["ept_new"])


Hoja 'results_export' reemplazada con 8 vendors exportados.
'to_adjust_now' se mantuvo intacta; last_executed_at actualizado en 9 filas.
Origen del ajuste:


,adjustment_origin,adjustment_scope,vendors
0,manual,vendor,8


,executed_at,wave,flag,adjustment_scope,franchise_id,franchise_name,vendor_code,store_name,ept_old,ept_change_pct,ept_new,ept_change_min
0,2026-09-04 17:27:09,<NA>,Manual modification,vendor,NaN,NaN,168677,Cocavi Delivery,13.2,0.5106,20,6.8
1,2026-09-04 17:27:09,<NA>,Manual modification,vendor,NaN,NaN,616172,Fullsabor,14.0,0.7857,25,11.0
2,2026-09-04 17:27:09,<NA>,Manual modification,vendor,NaN,NaN,452843,King Wins,10.0,1.0000,20,10.0
3,2026-09-04 17:27:09,<NA>,Manual modification,vendor,NaN,NaN,629758,La Papa De Moka´s.,16.0,0.2500,20,4.0
4,2026-09-04 17:27:09,<NA>,Manual modification,vendor,NaN,NaN,599640,Panaderia Y Pizzeria Vincenzo Pezzuoli,10.3,0.4569,15,4.7
5,2026-09-04 17:27:09,<NA>,Manual modification,vendor,NaN,NaN,542387,Rapaz Burgers - Tobalaba,37.2,-0.1946,30,-7.2
6,2026-09-04 17:27:09,<NA>,Manual modification,vendor,NaN,NaN,573294,Satoru Sushi 794,16.0,0.5625,25,9.0
7,2026-09-04 17:27:09,<NA>,Manual modification,vendor,NaN,NaN,597098,Sushi Del Brujo.,17.7,0.2995,23,5.3
